# КТ № 2 — Вариант 1. Изучение фикстур

Ноутбук показывает длину строки, сохранение текста и работу SQLite.
Положите этот файл рядом с `strings.py`, `pytest.ini` и папкой `tests`.
Примеры ниже выполнены, их вывод сохранён. Последняя ячейка запускает pytest при выполнении пользователем.

Для запуска в VS Code нужны расширения Python и Jupyter, а в выбранном окружении —
`python -m pip install -r requirements-notebook.txt`.
Откройте файл, выберите ядро Python этого окружения и выполните ячейки по порядку.


In [1]:
from pathlib import Path
import sys
import sqlite3
import tempfile
import subprocess
from strings import string_length, save_string

project = Path.cwd()
assert (project / "strings.py").is_file(), "Откройте ноутбук из папки variant_1"
print("Функции проекта подключены")


Функции проекта подключены


## 1. Длина строки
Проверяем пустую строку, пробелы, переносы и кириллицу. Ожидаемые длины заданы заранее.

In [2]:
examples = [("", 0), ("   ", 3), ("a\nb", 3), ("Привет", 6), ("a\r\nb", 4)]
for text, expected in examples:
    actual = string_length(text)
    assert actual == expected
    print(f"{text!r}: длина {actual}, ожидалось {expected}")


'': длина 0, ожидалось 0
'   ': длина 3, ожидалось 3
'a\nb': длина 3, ожидалось 3
'Привет': длина 6, ожидалось 6
'a\r\nb': длина 4, ожидалось 4


## 2. Сохранение строки
Записываем текст в UTF-8 и проверяем точность записи, затем перезаписываем файл. Временная папка удаляется автоматически.

In [3]:
with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / "example.txt"
    text = "Привет!\nИзучаем фикстуры."
    save_string(text, path)
    assert path.read_bytes() == text.encode("utf-8")
    print("Содержимое файла:")
    print(path.read_text(encoding="utf-8"))
    save_string("Новая строка", path)
    assert path.read_bytes() == "Новая строка".encode("utf-8")
    print("После перезаписи:", path.read_text(encoding="utf-8"))


Содержимое файла:
Привет!
Изучаем фикстуры.
После перезаписи: Новая строка


## 3. SQLite
Демонстрируем вставку, чтение и очистку таблицы. Настоящая pytest-фикстура показана в следующей ячейке.

In [4]:
with tempfile.TemporaryDirectory() as folder:
    database = Path(folder) / "example.sqlite3"
    connection = sqlite3.connect(database)
    try:
        connection.execute("CREATE TABLE strings (id INTEGER PRIMARY KEY, value TEXT NOT NULL)")
        connection.execute("INSERT INTO strings(value) VALUES (?)", ("Привет, SQLite!",))
        connection.commit()
        rows = connection.execute("SELECT value FROM strings").fetchall()
        assert rows == [("Привет, SQLite!",)]
        print("Записи:", rows)
    finally:
        try:
            connection.execute("DELETE FROM strings")
            connection.commit()
            remaining = connection.execute("SELECT COUNT(*) FROM strings").fetchone()[0]
            assert remaining == 0
            print("После очистки записей:", remaining)
        finally:
            connection.close()
            database.unlink()
    print("Файл базы удалён:", not database.exists())


Записи: [('Привет, SQLite!',)]
После очистки записей: 0
Файл базы удалён: True


## 4. Фикстуры проекта
`scope="function"` создаёт отдельную БД для каждого теста. `yield` передаёт соединение тесту; блок `finally` очищает ресурсы даже при ошибке теста. `tmp_path` предоставляет временный каталог.

In [5]:
print((project / "tests" / "conftest.py").read_text(encoding="utf-8"))


import sqlite3
import pytest

@pytest.fixture(params=["", "Первая\nВторая", "   ", "Привет", "a\r\nb"])
def text_value(request):
    return request.param

@pytest.fixture(scope="function")
def db(tmp_path):
    """Отдельная БД для теста; очистка выполняется и при падении теста."""
    path = tmp_path / "test.sqlite3"
    connection = sqlite3.connect(path)
    connection.execute("CREATE TABLE strings (id INTEGER PRIMARY KEY, value TEXT NOT NULL)")
    connection.commit()
    try:
        yield connection
    finally:
        try:
            connection.execute("DELETE FROM strings")
            connection.commit()
            assert connection.execute("SELECT COUNT(*) FROM strings").fetchone()[0] == 0
        finally:
            connection.close()
            path.unlink()



## 5. Запуск тестов
Ячейка запускает pytest в отдельном процессе с тем же Python, который выбран для ноутбука. Ожидаемый результат: **23 passed**. Эта ячейка здесь не выполнена; её вывод появится после запуска. При отсутствии pytest установите зависимости и повторите запуск.

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-v"],
    cwd=project, capture_output=True, text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, "Тесты не пройдены: посмотрите вывод выше"
